# Factor Validation

En este notebook se analizan las propiedades estadísticas y temporales de los factores construidos en la fase de *Feature Engineering*.

El objetivo es evaluar la calidad, estabilidad, persistencia y redundancia de cada factor antes de su utilización en modelos de selección de activos y construcción de carteras.

Los resultados obtenidos servirán para justificar las decisiones de preprocesamiento (winsorización, normalización, etc.) y la posible eliminación de factores redundantes.

In [39]:
import sys
from pathlib import Path
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np

# adds the root carpet of the project to the sys.path to allow importing modules from src
sys.path.append(str(Path.cwd().parent))

# autoreload configuration to automatically reload modules when they are modified
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Carga de Datos e inspección

In [3]:
prices = pd.read_parquet("../data/raw/sp500_prices.parquet")

# base variables
simple_returns = pd.read_parquet("../data/processed/simple_returns.parquet")
log_returns = pd.read_parquet("../data/processed/log_returns.parquet")
cumulative_returns = pd.read_parquet("../data/processed/cumulative_returns.parquet")

# style factors
momentum_12_1 = pd.read_parquet("../data/processed/momentum_12_1.parquet")
short_term_reversal = pd.read_parquet("../data/processed/short_term_reversal.parquet")
rolling_volatility = pd.read_parquet("../data/processed/rolling_volatility.parquet")
upside_volatility = pd.read_parquet("../data/processed/upside_volatility.parquet")
downside_volatility = pd.read_parquet("../data/processed/downside_volatility.parquet")
amihud_illiquidity = pd.read_parquet("../data/processed/amihud_illiquidity.parquet")


### 1.1 Información general por factor

In [38]:
factors = {
    "momentum_12_1": momentum_12_1,
    "short_term_reversal": short_term_reversal,
    "rolling_volatility": rolling_volatility,
    "upside_volatility": upside_volatility,
    "downside_volatility": downside_volatility,
    "amihud_illiquidity": amihud_illiquidity,
}

for name, df in factors.items():
    print(f"{name}")
    df.info()
    print("-" * 80)



momentum_12_1
<class 'pandas.DataFrame'>
DatetimeIndex: 3773 entries, 2010-01-04 to 2024-12-30
Columns: 499 entries, A to ZTS
dtypes: float64(499)
memory usage: 14.5 MB
--------------------------------------------------------------------------------
short_term_reversal
<class 'pandas.DataFrame'>
DatetimeIndex: 3773 entries, 2010-01-04 to 2024-12-30
Columns: 499 entries, A to ZTS
dtypes: float64(499)
memory usage: 14.5 MB
--------------------------------------------------------------------------------
rolling_volatility
<class 'pandas.DataFrame'>
DatetimeIndex: 3773 entries, 2010-01-04 to 2024-12-30
Columns: 499 entries, A to ZTS
dtypes: float64(499)
memory usage: 14.5 MB
--------------------------------------------------------------------------------
upside_volatility
<class 'pandas.DataFrame'>
DatetimeIndex: 3773 entries, 2010-01-04 to 2024-12-30
Columns: 499 entries, A to ZTS
dtypes: float64(499)
memory usage: 14.5 MB
------------------------------------------------------------------

### 1.2 Matriz de cobertura por cada factor

In [31]:
from src.analysis.factor_diagnostics import get_warmup_trading_days 

factors = {
    "momentum_12_1": momentum_12_1,
    "short_term_reversal": short_term_reversal,
    "rolling_volatility": rolling_volatility,
    "upside_volatility": upside_volatility,
    "downside_volatility": downside_volatility,
    "amihud_illiquidity": amihud_illiquidity,
}

summary = pd.DataFrame(
    [
        {
            "factor": name,
            "total_values": df.size,
            "valid_values": df.notna().sum().sum(),
            "nan_percentage": df.isna().sum().sum() / df.size * 100,
            "global_start_date": (
                df.index.get_level_values("Date")[df.notna().any(axis=1)].min()
            ),
            "global_end_date": (
                df.index.get_level_values("Date")[df.notna().any(axis=1)].max()
            ),
            "warmup_trading_days": get_warmup_trading_days(df),
        }
        for name, df in factors.items()
    ]
).set_index("factor")

summary["nan_percentage"] = summary["nan_percentage"].round(2)

summary

,total_values,valid_values,nan_percentage,global_start_date,global_end_date,warmup_trading_days
factor,,,,,,
momentum_12_1,1882727,1644211,12.67,2011-01-03,2024-12-30,252
short_term_reversal,1882727,1759361,6.55,2010-02-03,2024-12-30,21
rolling_volatility,1882727,1644211,12.67,2011-01-03,2024-12-30,252
upside_volatility,1882727,1747377,7.19,2010-02-16,2024-12-30,29
downside_volatility,1882727,1745649,7.28,2010-02-12,2024-12-30,28
amihud_illiquidity,1882727,1757181,6.67,2010-01-26,2024-12-30,15


## 2. Análisis de Cobertura y Calidad de Datos (Data Quality)
   - Porcentaje de valores nulos (NaNs) por factor y por fecha.
   - Nº de activos válidos en la cross-section a lo largo del tiempo.
   - Identificación de sesgo de supervivencia o inicio de series (Warm-up period por rolling windows).

Procedimiento:

Calcular la serie temporal de activos con datos válidos (count() no nulo) por fecha para cada factor.

Identificar el período de warm-up inicial (días sin datos al principio de la serie debido a las ventanas rodantes, como los 252 días necesarios para el 12-1 Momentum).

Contar el porcentaje de NaNs global y la evolución diaria de cobertura cross-sectional.

Qué comparar:
Comparar el "goteo" de NaNs entre factores de ventana corta (ej. Reversal) frente a factores de ventana larga (Momentum).

Conclusión a la que debes llegar:
Definir la fecha de inicio oficial limpia del dataset (descartando la ventana de warm-up común) y verificar que no hay "agujeros" de cobertura por fallos de API o deslistados de acciones no tratados.

In [40]:
from src.analysis.factor_diagnostics import count_nans_after_first_valid

nans_after_first_valid = {
    name: count_nans_after_first_valid(df)
    for name, df in factors.items()
}

nans_after_first_valid

{'momentum_12_1': np.int64(112768),
 'short_term_reversal': np.int64(112887),
 'rolling_volatility': np.int64(112768),
 'upside_volatility': np.int64(120879),
 'downside_volatility': np.int64(123106),
 'amihud_illiquidity': np.int64(118061)}

## 3. Análisis de Distribución (Univariado)
   - Tabla de Estadísticos Descriptivos Consolidados (Mean, Std, Min, P25, P50, P75, Max, Skew, Kurtosis).
   - Visualización de distribuciones (KDE / Histogramas / QQ Plot frente a Normal) para detectar colas pesadas (Heavy-tails).
   - Identificación de asimetría crítica (ej. Amihud, Volatilidad).
   - Outliers

Procedimiento:

Generar la tabla consolidada de estadísticos globales (Media, Std, Min, P25, P50, P75, Max, Skewness y Kurtosis).

Pintar la distribución de cada factor mediante gráficos de densidad (KDE), histogramas y un Q-Q Plot comparado contra la distribución Normal estándar.

Qué comparar:
Comparar la mediana frente a la media para medir el sesgo, y observar las colas en los Q-Q plots para identificar heavy-tails (especialmente en Amihud y volatilidades).

Conclusión a la que debes llegar:
Determinar formalmente qué factores son no-normales o tienen asimetría crítica. Esto justificará por qué en el notebook 05 será obligatorio aplicar transformaciones como el logaritmo (para comprimir la cola derecha) y Winsorization.

## 4. Comportamiento Temporal y Estabilidad Cross-Sectional
   - Serie temporal de la media y mediana cross-sectional por factor.
   - Dispersión Cross-Sectional (Desviación estándar / Rango Intercuartílico diario) para verificar que el factor mantenga varianza a lo largo de los ciclos de mercado.
   - Coeficiente de variación transversal.

Procedimiento:

Para cada fecha $t$, calcular la media, la mediana, la desviación estándar y el Rango Intercuartílico (IQR = P75 - P25) del universo de activos.

Graficar las series temporales de estas métricas y calcular el Coeficiente de Variación transversal ($\text{CV} = \frac{\text{Std}}{\text{Mean}}$).

Qué comparar:
Comparar el comportamiento del factor en períodos de calma frente a momentos de alta volatilidad (ej. crisis de 2008 o 2020) para ver si la varianza del factor colapsa a cero o explota.

Conclusión a la que debes llegar:
Validar que el factor mantiene suficiente dispersión transversal en cualquier entorno macroeconómico. Si un factor pierde la dispersión (todos los activos valen lo mismo), deja de servir para ordenar la cartera.

## 5. Persistencia y Memoria Temporal
   - Autocorrelación de primer orden (Lag-1) por activo/factor.
   - Matriz de Autocorrelación Temporal (t vs t+1, t+5, t+21) -> Relevante para decidir la frecuencia de rebalanceo de la cartera. Podemos meter también la correlación cross sectional entre t y t+k para k = 1 dia, 5 dias y 21 dias (para persistencia del ranking). 

Procedimiento:

Calcular la autocorrelación de primer orden ($\text{Lag-1}$) de cada factor para cada activo y extraer la mediana del universo.

Calcular la correlación del factor en $t$ frente a sus propios valores desplazados en $t+1$, $t+5$ (1 semana) y $t+21$ (1 mes).

Qué comparar:
La tasa de decaimiento de la autocorrelación entre factores rápidos (ej. Short-Term Reversal) y factores lentos (ej. 12-1 Momentum o Amihud).

Conclusión a la que debes llegar:
Establecer la frecuencia lógica de rebalanceo de la cartera. Si un factor pierde toda su memoria a los 5 días, no puedes rebalancear mensualmente; si la mantiene intacta a 21 días, exige menos rotación y menos costes de transacción.

## 6. Redundancia y Multicolinealidad (Análisis Multivariado)
   - Matriz de Correlación Cross-Sectional Promedio (Pearson para relaciones lineales, Spearman para rango/monotonía).
   - Dendrograma de Jerarquía de Factores (Hierarchical Clustering) para agrupar factores redundantes.
   - VIF (Variance Inflation Factor) preliminar para detectar multicolinealidad severa.
   - Comparativa explícita de subgrupos con alta correlación esperada:
     * Retornos: Simple vs Log vs Cumulative.
     * Volatilidad: Rolling Total vs Upside vs Downside.
     * Momento/Reversal: Short-Term Reversal vs 12-1 Momentum.
  - Preliminary Predictive Power (IC preliminar): podemos calcular Spearman entre Factor(t) y Return(t+21) para cada fecha, obteneos una serie temporal para cada IC y calulamos Mean IC, Std IC y ICIR.  

Procedimiento:

Calcular la matriz de correlación Cross-Sectional diaria y promediarla a lo largo del tiempo (tanto Pearson como Spearman por rangos).

Aplicar un Hierarchical Clustering sobre la matriz de distancias ($1 - \vert{}\rho\vert{}$) y dibujar el Dendrograma.

Calcular el VIF (Variance Inflation Factor) para los 6 factores.

Sub-análisis de redundancia:
  - Retornos (Simple vs Log vs Cumulative).
  - Volatilidades (Total vs Upside vs Downside).
  - Momento vs Reversal.

Qué comparar:
Diferencias entre Pearson y Spearman (si Spearman es mucho mayor, indica relación no lineal pero monótona). Analizar VIFs superiores a 5 o 10.

Conclusión a la que debes llegar:
Identificar objetivamente qué parejas de factores están midiendo la misma señal y cuál de ellas aporta información limpia o es candidata a ser eliminada.


## 7. Conclusiones y Plan de Acción
   - Lista de factores candidatos a eliminación / fusión.
   - Resumen de decisiones previo al Pipeline de Preprocesamiento (Winsorize/Z-score). Lo podemos llamar Preprocessing Decisions

Procedimiento:
Sintetizar los hallazgos de las secciones anteriores en una tabla final de decisiones metodológicas.

Qué comparar:
Cruzar la utilidad teórica del factor con los hallazgos empíricos (asimetría, autocorrelación, VIF).

Conclusión a la que debes llegar:
Entregar una lista cerrada de los factores finales aprobados y el roadmap explícito para el notebook 05:
   - Factores eliminados por redundancia extrema.
   - Factores retenidos y sus requerimientos de preprocesamiento (ej. Amihud $\rightarrow$ Logaritmo + Winsorize + Z-score; Momentum $\rightarrow$ Z-score directo).